# Limpieza de unidades_samur.csv

Catálogo de las unidades (vehículos) del SAMUR: id, código, tipo de unidad, distrito de la base, año de adquisición, kilómetros recorridos y estado.

El CSV trae 60 filas y se queda con 60: no hay copias idénticas que quitar (todas tienen `unidad_id` distinto). El identificador bueno es `unidad_id` (no se repite); el `codigo` se repite y no sirve para cruzar, como ya vimos en `cruces_datasets.ipynb`.

El CSV original no se toca: el resultado se guarda aparte en `unidades_samur_limpio.csv`.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('unidades_samur.csv')
if not ruta.exists():
    ruta = Path('../data/unidades_samur.csv')
uni = pd.read_csv(ruta)
uni.head()

,unidad_id,codigo,tipo_unidad,base_distrito,año_adquisicion,km_recorridos,estado
0,1,SAMUR-001,UVI,HORTALEZA,2021,139286,operativa
1,2,SAMUR-002,Moto,FUENCARRAL,2012,225222,operativa
2,3,SAMUR-003,VIR,LATINA,2010,251639,mantenimiento
3,4,SAMUR-004,S.V.B.,CIUDAD LINEAL,2016,52379,Operativa
4,5,SAMUR-005,U.V.I.,TETUAN,2014,248981,OK


## Revisión inicial

Miramos los datos antes de tocar nada: cuántas filas hay, dónde están los huecos y qué valores raros aparecen.

In [2]:
print(uni.shape)              # 60 filas
print()
print(uni.isna().sum())       # huecos en tipo_unidad y base_distrito
print()
print(f"unidad_id repetidos: {uni['unidad_id'].duplicated().sum()}")   # 0: el id es la clave buena
print(f"Codigos repetidos: {uni['codigo'].duplicated().sum()}")        # el codigo NO es clave
print(f"Filas duplicadas enteras: {uni.duplicated().sum()}")           # 0: no hay copias exactas
print()
print(f"Año min/max: {uni['año_adquisicion'].min()} / {uni['año_adquisicion'].max()}")
print(f"Km min/max: {uni['km_recorridos'].min()} / {uni['km_recorridos'].max()}")
print()
print("Tipos de unidad escritos de distintas formas:")
print(sorted(uni['tipo_unidad'].dropna().unique()))
print()
print("Estados escritos de distintas formas:")
print(sorted(uni['estado'].dropna().unique()))

(60, 7)

unidad_id          0
codigo             0
tipo_unidad        8
base_distrito      5
año_adquisicion    0
km_recorridos      0
estado             0
dtype: int64

unidad_id repetidos: 0
Codigos repetidos: 10
Filas duplicadas enteras: 0

Año min/max: 2010 / 2029
Km min/max: -3134 / 1447628

Tipos de unidad escritos de distintas formas:
['JEFATURA', 'Jefatura', 'MOTO', 'Moto', 'S.V.B.', 'SVB', 'U.V.I.', 'UVI', 'VIR', 'moto', 'uvi']

Estados escritos de distintas formas:
['BAJA', 'Baja', 'MANT', 'OK', 'OPERATIVA', 'Operativa', 'baja', 'inactiva', 'mantenimiento', 'operativa']


## Códigos repetidos

El `codigo` (SAMUR-XXX) se repite en 10 parejas:

- 8 parejas se contradicen en algo (tipo, estado o kilómetros distintos).
- 2 parejas (SAMUR-032 y SAMUR-033) son iguales en todo menos en el `unidad_id`.

En activaciones quitábamos las copias idénticas enteras; aquí no hay ninguna (todas cambian al menos en el id). Puede que esas 2 parejas sean la misma unidad registrada dos veces, pero no podemos saberlo, así que se conservan todas las filas y se documenta, igual que hicimos en activaciones con los ids repetidos que no coincidían. Lo importante para los cruces: unir por `unidad_id`, nunca por `codigo`.

In [3]:
repetidos = uni[uni.duplicated('codigo', keep=False)].sort_values(['codigo', 'unidad_id'])
print(f"Filas con codigo repetido: {len(repetidos)} (10 parejas)")
print(repetidos.to_string(index=False))
print()

# Comprobamos si alguna pareja es identica en todo menos el id
sin_id = uni.drop(columns=['unidad_id'])
copias = uni[sin_id.duplicated(keep=False)]
print(f"Filas identicas salvo unidad_id: {len(copias)} -> se conservan (no sabemos si son la misma unidad)")

Filas con codigo repetido: 20 (10 parejas)
 unidad_id    codigo tipo_unidad base_distrito  año_adquisicion  km_recorridos        estado
         4 SAMUR-004      S.V.B. CIUDAD LINEAL             2016          52379     Operativa
        58 SAMUR-004    Jefatura CIUDAD LINEAL             2016        1447628     operativa
         5 SAMUR-005      U.V.I.        TETUAN             2014         248981            OK
        53 SAMUR-005         UVI        TETUAN             2014         248981 mantenimiento
        10 SAMUR-010        Moto        TETUAN             2017          52622     operativa
        59 SAMUR-010         VIR        TETUAN             2017          -3089     operativa
        15 SAMUR-015        MOTO    FUENCARRAL             2023         168885     operativa
        56 SAMUR-015        Moto    FUENCARRAL             2023         168885          Baja
        24 SAMUR-024         uvi           NaN             2013          98952     operativa
        60 SAMUR-024       

## Tipo de unidad y estado

El tipo de unidad viene con mayúsculas, minúsculas y puntos mezclados (`UVI`, `U.V.I.`, `uvi`). Quitando puntos y poniendo mayúsculas quedan las 5 categorías reales (`UVI`, `SVB`, `VIR`, `MOTO`, `JEFATURA`). Es un cambio de escritura, no de significado, así que no hace falta ninguna lista de equivalencias. Las 8 filas sin tipo se quedan vacías: no sabemos qué unidad son y no lo inventamos.

El estado también se normaliza por escritura (mayúsculas, sin tildes), y `MANT` pasa a `MANTENIMIENTO` porque es el principio de una única categoría, la misma regla que usamos con los nombres de distrito. **`OK` se deja como categoría propia:** seguramente quiera decir "operativa", pero no podemos comprobarlo y no inventamos la equivalencia. Queda documentado aquí y en el README de estas limpiezas.

In [4]:
def normalizar_texto(s):
    if pd.isna(s):
        return None
    s = unicodedata.normalize('NFD', str(s).strip().upper())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return ' '.join(s.split())

uni['tipo_unidad'] = uni['tipo_unidad'].map(lambda s: normalizar_texto(s).replace('.', '') if normalizar_texto(s) else None)
print("Tipos tras normalizar:", sorted(uni['tipo_unidad'].dropna().unique()))
print(f"Filas sin tipo (quedan vacías): {uni['tipo_unidad'].isna().sum()}")
print()

uni['estado'] = uni['estado'].map(normalizar_texto)

def completar_categoria(t):
    # si el texto es el principio de una unica categoria, se completa (MANT -> MANTENIMIENTO)
    candidatos = [c for c in ('BAJA', 'INACTIVA', 'MANTENIMIENTO', 'OPERATIVA') if c.startswith(t)]
    if len(candidatos) == 1:
        return candidatos[0]
    return t   # OK no es principio de ninguna: se queda como esta

uni['estado'] = uni['estado'].map(completar_categoria)
print("Estados tras normalizar:", sorted(uni['estado'].unique()))
print()
print(uni['estado'].value_counts())

Tipos tras normalizar: ['JEFATURA', 'MOTO', 'SVB', 'UVI', 'VIR']
Filas sin tipo (quedan vacías): 8

Estados tras normalizar: ['BAJA', 'INACTIVA', 'MANTENIMIENTO', 'OK', 'OPERATIVA']

estado
OPERATIVA        44
MANTENIMIENTO     6
BAJA              6
OK                3
INACTIVA          1
Name: count, dtype: int64


## Distrito de la base

Los distritos vienen como `FUENCARRAL` o `VALLECAS`. Se normaliza con la misma regla de nombres de siempre: mayúsculas, sin tildes, sin artículo, y un texto solo se completa si es el principio de un único distrito oficial. Así `FUENCARRAL` pasa a `FUENCARRAL-EL PARDO`, que es el único distrito oficial que empieza por ahí.

**`VALLECAS` se queda como `VALLECAS`, sin asignar.** Es el principio de dos distritos (Puente y Villa de Vallecas) y no tenemos ninguna fuente que diga a cuál pertenece cada unidad. En `cruces_datasets.ipynb` las contamos en Puente de Vallecas porque es el único de los dos con base en el catálogo de bases, pero eso es una inferencia (el catálogo podría estar incompleto), no un dato: lo dejamos explícito en vez de repetirla aquí. Son 10 unidades afectadas, así que en `tabla_distritos.csv` el número de unidades de Puente de Vallecas depende de esa suposición; queda pendiente de confirmar con Bluetab.

A 5 unidades les falta el distrito de la base: se marcan como `NO CONSTA`, igual que en activaciones, porque no sabemos de qué base son.

In [5]:
OFICIALES = ["CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
             "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
             "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA", "VILLAVERDE",
             "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"]

def normalizar_distrito(nombre):
    t = normalizar_texto(nombre)
    if t is None:
        return 'NO CONSTA'
    for articulo in ('LA ', 'EL ', 'LOS ', 'LAS '):
        if t.startswith(articulo):
            t = t[len(articulo):]
    if t in OFICIALES:
        return t
    candidatos = [o for o in OFICIALES if o.startswith(t)]
    if len(candidatos) == 1:
        return candidatos[0]
    return t   # no encaja con uno solo (VALLECAS): se conserva el texto normalizado, sin asignar

uni['base_distrito'] = uni['base_distrito'].map(normalizar_distrito)
print(f"Unidades sin distrito (NO CONSTA): {(uni['base_distrito'] == 'NO CONSTA').sum()}")
print(f"Unidades con VALLECAS a secas (ambiguo Puente/Villa): {(uni['base_distrito'] == 'VALLECAS').sum()}")
print()
print(uni['base_distrito'].value_counts())

Unidades sin distrito (NO CONSTA): 5
Unidades con VALLECAS a secas (ambiguo Puente/Villa): 10

base_distrito
LATINA                 11
TETUAN                 11
VALLECAS               10
FUENCARRAL-EL PARDO     6
CIUDAD LINEAL           5
CARABANCHEL             5
NO CONSTA               5
HORTALEZA               4
CENTRO                  3
Name: count, dtype: int64


## Año de adquisición imposible

Los datos nos los entregaron con el proyecto en septiembre de 2026 (correo de Bluetab del 15/09/2026). Dos unidades tienen año de adquisición 2029: es posterior a la propia entrega del dataset (y a hoy) y ya traen kilómetros registrados, así que es imposible; es un error de registro y pasa a vacío, porque no sabemos el año real.

Otras dos tienen 2026: no tenemos ninguna evidencia para invalidarlo. La fecha de entrega del dataset no fija la fecha de observación del catálogo (puede ser más reciente), así que se conservan tal cual. Y tampoco asumimos lo contrario: que el catálogo sea una foto de 2024 solo porque las activaciones sean de 2024.

In [6]:
print("Años posteriores a 2026 (posteriores a la entrega del dataset, imposibles):")
print(uni.loc[uni['año_adquisicion'] > 2026, ['unidad_id', 'codigo', 'año_adquisicion']].to_string(index=False))

uni.loc[uni['año_adquisicion'] > 2026, 'año_adquisicion'] = np.nan
uni['año_adquisicion'] = uni['año_adquisicion'].astype('Int64')  # entero que admite vacíos
print()
print(f"Años pasados a vacío: {uni['año_adquisicion'].isna().sum()}")
print(f"Rango tras limpiar: {uni['año_adquisicion'].min()} - {uni['año_adquisicion'].max()}")

Años posteriores a 2026 (posteriores a la entrega del dataset, imposibles):
 unidad_id    codigo  año_adquisicion
         8 SAMUR-008             2029
        51 SAMUR-047             2029

Años pasados a vacío: 2
Rango tras limpiar: 2010 - 2026


## Kilómetros imposibles

Cinco unidades tienen kilómetros negativos (-3.134, -210, -52, -1.970 y -3.089), que es imposible. Pasan a vacío: no sabemos el valor real y no lo inventamos.

Los valores altos se quedan como están: el máximo es 1.447.628 km, que es mucho pero no es imposible para un vehículo de emergencias, y no tenemos ninguna regla que lo descarte.

In [7]:
print("Kilometros negativos (imposibles):")
print(uni.loc[uni['km_recorridos'] < 0, ['unidad_id', 'codigo', 'km_recorridos']].to_string(index=False))

uni.loc[uni['km_recorridos'] < 0, 'km_recorridos'] = np.nan
uni['km_recorridos'] = uni['km_recorridos'].astype('Int64')
print()
print(f"Km pasados a vacío: {uni['km_recorridos'].isna().sum()}")
print(f"Km min/max tras limpiar: {uni['km_recorridos'].min()} / {uni['km_recorridos'].max()}")

Kilometros negativos (imposibles):
 unidad_id    codigo  km_recorridos
        11 SAMUR-011          -3134
        20 SAMUR-020           -210
        54 SAMUR-025            -52
        55 SAMUR-044          -1970
        59 SAMUR-010          -3089

Km pasados a vacío: 5
Km min/max tras limpiar: 52379 / 1447628


## Estadísticos y guardado

Las 60 filas quedan ordenadas por `unidad_id`. Miramos los estadísticos finales y guardamos.

Recordatorio para los cruces: unir por `unidad_id`, no por `codigo`, que se repite. Y ojo: las 10 unidades con `VALLECAS` no están asignadas a Puente ni a Villa.

In [8]:
uni = uni.sort_values('unidad_id').reset_index(drop=True)
# Mismo orden de columnas que el original
uni = uni[['unidad_id', 'codigo', 'tipo_unidad', 'base_distrito',
           'año_adquisicion', 'km_recorridos', 'estado']]
uni.to_csv('../data_clean/unidades_samur_limpio.csv', index=False)
print(uni.shape)   # (60, 7)
print()
print("Huecos que quedan (los valores que no conocemos):")
print(uni.isna().sum()[uni.isna().sum() > 0])
print()
uni.describe().round(2)

(60, 7)

Huecos que quedan (los valores que no conocemos):
tipo_unidad        8
año_adquisicion    2
km_recorridos      5
dtype: int64



,unidad_id,año_adquisicion,km_recorridos
count,60.00,58.0,55.0
mean,30.50,2016.31,230571.67
std,17.46,4.5,271075.4
min,1.00,2010.0,52379.0
25%,15.75,2013.0,100020.5
50%,30.50,2016.0,186315.0
75%,45.25,2019.75,243496.0
max,60.00,2026.0,1447628.0
